# E7 — End-to-end composition and concentration

Predict gas presence and nominal concentrations directly from sensor signals. Evaluate a future day within campaign 130 and a later campaign 146 using identical trained models, without oracle labels at inference.

In [ ]:
from pathlib import Path
from io import StringIO
import os, json, sys, hashlib, copy, random, time, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler
from scipy.special import expit

ROOT=Path.cwd().resolve()
SRC=next((q for p in (ROOT,*ROOT.parents) for q in (p,p/"src") if (q/"esdata").is_dir()),None)
if SRC is None: raise FileNotFoundError("Run from the repository root or src/")
OUT=SRC/"outputs"/"E7_end_to_end_v1";OUT.mkdir(parents=True,exist_ok=True)
DATA_DIR=Path(os.getenv("ESMELL_DATA_DIR",str(SRC.parent/"data"/"raw")))
CHANNELS=("R1","R2","R11","R12","R16","R17")
GASES=("acetone","isopropanol","pentane")
BITS=np.array([[int(i>>j&1) for j in range(3)] for i in range(8)],dtype=np.uint8)
SEEDS=(42,43,44)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_EPOCHS,PATIENCE,BATCH=80,14,64
LAMBDA=0.5                       # joint loss weight; fixed, NOT tuned on either test
RUN_ROCKET=True                  # require sktime; False is only a partial debugging run
RUN_NEURAL=True                  # leave True for a complete paper run

def sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for block in iter(lambda:f.read(4<<20),b""):h.update(block)
    return h.hexdigest()
def seeded(s):
    random.seed(s);np.random.seed(s);torch.manual_seed(s)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

print("device",DEVICE,"output",OUT)

In [ ]:
# Reuse E5/E6's identical per-sample preprocessing; verify archive+source signature.
STEMS={130:"curiosity-2v15.130.0",146:"curiosity-2v15.146.0"}
assert DATA_DIR.is_dir(),f"Set ESMELL_DATA_DIR to archive directory: {DATA_DIR}"
ARCHIVES={}
for card,stem in STEMS.items():
    matches=list(DATA_DIR.glob(stem+".tar*.gz"))
    if len(matches)!=1:raise FileNotFoundError(f"Expected exactly one archive for {stem}: {matches}")
    ARCHIVES[card]=matches[0]
code_hashes={str(p.relative_to(SRC)):sha256(p) for p in (SRC/"esdata").rglob("*.py")}
assert code_hashes
signature_payload=json.dumps(dict(
    archive_hashes={c:sha256(p) for c,p in ARCHIVES.items()},
    code_hashes=code_hashes,channels=CHANNELS,
    profile="DefaultCSV+Simplify+SelectR+Cycles+CoolHeat3eps.2+CanonicalTempAlign200(175,390)+Subtraction1,-3,5"),sort_keys=True)
SIGNATURE=hashlib.sha256(signature_payload.encode()).hexdigest()

def read_cache(path):
    if not path.exists():return None
    with np.load(path,allow_pickle=False) as z:
        if str(z["signature"].item())!=SIGNATURE:return None
        meta=pd.read_json(StringIO(str(z["metadata_json"].item())),orient="records")
        X=z["X"].astype(np.float32)
    meta["time"]=pd.to_datetime(meta["time"])
    usable=meta[meta.split.ne("excluded")].sort_values(["card","time"]).reset_index(drop=True)
    assert len(usable)==len(X) and X.shape[1:]==(12,200)
    return usable,X

cache_paths=[OUT/"signals_cache.npz",
             SRC/"outputs"/"E5_diagnostic_v2"/"signals_cache.npz",
             SRC/"outputs"/"E6_concentration_v1"/"signals_cache.npz"]
loaded=next((z for p in cache_paths if (z:=read_cache(p)) is not None),None)
if loaded is not None:
    D,X=loaded
    print("Loaded signature-verified E5/E6 signal cache")
else:
    sys.path.insert(0,str(SRC))
    from esdata import EsmellDataset,Chain
    from esdata.transforms import (DefaultCSVPreprocessing,SimplifyLabels,SelectRChannels,
        UserLabelCycleSegments,CoolingHeatingSegments,CanonicalTempAlign,CanonicalSubtractionFeatures)
    rows=[];arrays={}
    for card,path in ARCHIVES.items():
        ds=EsmellDataset(root_path=DATA_DIR,subfolders=path.name,
                         csv_repair="strict",archive_cache="disk",preload=True)
        pipe=ds>>Chain(DefaultCSVPreprocessing(),SimplifyLabels(),
            SelectRChannels(include=CHANNELS),UserLabelCycleSegments(),
            CoolingHeatingSegments(temp_col="T_mean",smooth_window=3,eps=.2),
            CanonicalTempAlign(n_grid=200,temp_range=(175.,390.),normalize=False,conductance=True),
            CanonicalSubtractionFeatures(features=((1,-3,5),((1,-3,5),(0,-2,4)))))
        for sid in ds.ids:
            m=ds.metadata(sid);t=pd.Timestamp(m["datetime"])
            split=("test" if card==146 else "train" if t<pd.Timestamp("2026-09-28")
                   else "validation" if t<pd.Timestamp("2026-09-29") else "excluded")
            rows.append(dict(card=card,sample_id=str(sid),filename=m["filename"],
                             time=t,split=split,recipe=str(m["classname"]).lower()))
            if split!="excluded":
                a=np.asarray(pipe.features(sid),dtype=np.float32)
                if a.shape!=(12,200) or not np.isfinite(a).all():raise ValueError((sid,a.shape))
                arrays[(card,str(sid))]=a
    meta=pd.DataFrame(rows)
    D=meta[meta.split.ne("excluded")].sort_values(["card","time"]).reset_index(drop=True)
    X=np.stack([arrays[(r.card,r.sample_id)] for r in D.itertuples()])
    np.savez_compressed(OUT/"signals_cache.npz",signature=np.array(SIGNATURE),
        metadata_json=np.array(meta.to_json(orient="records",date_format="iso")),X=X)

assert X.shape==(3477,12,200) and np.isfinite(X).all()
assert D.filename.is_unique and not D[["card","sample_id"]].duplicated().any()
print("Signals:",X.shape)

## 1. Labels and evaluation cohorts

Hold out 28 September and all of campaign 146. Select models on grouped hours from 27 September and fit on the remaining earlier records.

In [ ]:
PAT=re.compile(r"^(acetone|isopropanol|pentane)\[([0-9]+(?:\.[0-9]+)?)ppb\]$")
def parse_recipe(s):
    c=np.full(3,np.nan,dtype=float)
    if s=="air":return c
    for part in s.lower().split("+"):
        m=PAT.fullmatch(part.strip())
        if m is None:raise ValueError("Unexpected analyte/recipe: "+s)
        j=GASES.index(m.group(1))
        if np.isfinite(c[j]):raise ValueError("Duplicate analyte: "+s)
        c[j]=float(m.group(2))
    return c

C=np.vstack([parse_recipe(str(s)) for s in D.recipe])
P=np.isfinite(C).astype(np.uint8)
Y=np.where(P,C,0.)                 # zero ONLY for explicitly absent gases
LOG=np.log10(np.where(P,C,1.))
COMP=(P*np.array([1,2,4])).sum(1).astype(int)
CARD=D.card.to_numpy()
DT=pd.to_datetime(D.time)
SEP27=np.flatnonzero((CARD==130)&(DT.dt.date==pd.Timestamp("2026-09-27").date()))
# These six complete hourly blocks were frozen from fold 0 of a grouped,
# stratified 4-fold partition of Sep 27 (seed 42, sklearn 1.8).
# Explicit hours ensure identical validation records across sklearn versions.
VAL_HOURS={
    "2026-09-27T03:00","2026-09-27T05:00","2026-09-27T08:00",
    "2026-09-27T18:00","2026-09-27T21:00","2026-09-27T23:00"
}
VA=SEP27[DT.iloc[SEP27].dt.strftime("%Y-%m-%dT%H:00").isin(VAL_HOURS).to_numpy()]
TR=np.flatnonzero((CARD==130)&(DT<pd.Timestamp("2026-09-28"))&~np.isin(np.arange(len(D)),VA))
TEST130=np.flatnonzero((CARD==130)&(DT>=pd.Timestamp("2026-09-28"))
                      &(DT<pd.Timestamp("2026-09-29")))
TEST146=np.flatnonzero(CARD==146)
SPLITS={"train":TR,"validation":VA,"within130":TEST130,"future146":TEST146}

assert len(SEP27)==293 and len(TEST130)==152 and len(TEST146)==916
assert len(VA)==71 and len(TR)==2338
assert all(set(COMP[idx])==set(range(8)) for idx in SPLITS.values())
assert max(DT.iloc[np.r_[TR,VA]])<min(DT.iloc[TEST130])
assert max(DT.iloc[np.r_[TR,VA]])<min(DT.iloc[TEST146])
assert not (set(DT.iloc[TR].dt.floor("h")) & set(DT.iloc[VA].dt.floor("h")))
assert not any(np.intersect1d(SPLITS[a],SPLITS[b]).size for a,b in
               [("train","validation"),("train","within130"),("train","future146"),
                ("validation","within130"),("validation","future146")])
assert (P.sum(1)>=2).sum()>0 and np.isfinite(C[P==1]).all()

# Recipe canonicalization makes concentration-order permutations equivalent.
def recipe_key(row):
    parts=[f"{g}[{float(row[j]):g}ppb]" for j,g in enumerate(GASES) if np.isfinite(row[j])]
    return "+".join(parts) if parts else "air"
D["canonical_recipe"]=[recipe_key(row) for row in C]
D["protocol_split"]="excluded"
for s,idx in SPLITS.items(): D.loc[idx,"protocol_split"]=s
D.to_csv(OUT/"split_manifest.csv",index=False)

for name,idx in SPLITS.items():
    ct=pd.Series(COMP[idx]).value_counts().reindex(range(8),fill_value=0)
    print(f"{name:11s} n={len(idx):4d} mixtures={int((P[idx].sum(1)>=2).sum()):3d}"
          f" composition counts={ct.tolist()}")

## 2. Metrics and controls

Report macro-averaged normalized ppb MAE across eight compositions, composition macro recall, mixture exact match, and exact decomposition into false-negative, false-positive, and correctly detected component errors.

In [ ]:
def stats192(a):
    cols=[a.mean(2),a.std(2),a.min(2),a.max(2),
          np.median(a,axis=2),np.quantile(np.abs(a),.9,axis=2)]
    cols.extend(z.mean(2) for z in np.array_split(a,10,axis=2))
    return np.concatenate(cols,axis=1).astype(np.float32)

F=stats192(X)
assert F.shape==(len(D),192)
SCALE=Y[TR].max(0)
LO=np.array([LOG[TR,j][P[TR,j]==1].min() for j in range(3)])
HI=np.array([LOG[TR,j][P[TR,j]==1].max() for j in range(3)])
assert (SCALE>0).all()
MEAN=X[TR].mean((0,2),keepdims=True)
STD=np.maximum(X[TR].std((0,2),keepdims=True),1e-6)
XN=((X-MEAN)/STD).astype(np.float32)
TRAIN_COUNTS=np.bincount(COMP[TR],minlength=8)
W_CLS=(len(TR)/(8*TRAIN_COUNTS[COMP])).astype(np.float32)
W_CLS/=W_CLS[TR].mean()

# Per-gas, split positive supervision weight equally between pure/mixed calibration.
W_REG=np.zeros_like(Y,dtype=np.float32)
for j in range(3):
    pure=TR[(P[TR,j]==1)&(P[TR].sum(1)==1)]
    mix=TR[(P[TR,j]==1)&(P[TR].sum(1)>=2)]
    assert len(pure) and len(mix)
    W_REG[pure,j]=0.5/len(pure)
    W_REG[mix,j]=0.5/len(mix)
# The relative scale within minibatches is normalized by sum(weight) in the loss.

def pred_ppb(presence,log_pred):
    pres=np.asarray(presence,bool)
    log_pred=np.asarray(log_pred,float)
    bounded=np.clip(log_pred,LO,HI)
    out=np.where(pres,np.power(10.,bounded),0.)
    assert np.isfinite(out).all() and (out>=0).all()
    return out

def class_metrics(idx,pred_presence):
    truth=P[idx].astype(int);z=np.asarray(pred_presence,int)
    true_comp=COMP[idx]
    pred_comp=(z*np.array([1,2,4])).sum(1)
    recall=[float(np.mean(pred_comp[true_comp==k]==k)) for k in range(8)]
    mix=truth.sum(1)>=2
    double=truth.sum(1)==2
    triple=truth.sum(1)==3
    bit_bacc=[]
    for j in range(3):
        pos=truth[:,j]==1;neg=~pos
        bit_bacc.append(.5*(np.mean(z[pos,j]==1)+np.mean(z[neg,j]==0)))
    return dict(macro8=float(np.mean(recall)),
                exact_all=float(np.mean(np.all(z==truth,axis=1))),
                exact_mix=float(np.mean(np.all(z[mix]==truth[mix],axis=1))),
                exact_double=float(np.mean(np.all(z[double]==truth[double],axis=1))),
                exact_triple=float(np.mean(np.all(z[triple]==truth[triple],axis=1))),
                bit_bacc=float(np.mean(bit_bacc)))

def e2e_metrics(idx,z,chat):
    p=P[idx].astype(bool);zz=np.asarray(z,bool);real=Y[idx]
    err=np.abs(chat-real)
    fn=p&~zz;fp=~p&zz;tp=p&zz
    decomposition=[float(err[q].sum()) for q in (fn,fp,tp)]
    total=float(err.sum())
    assert np.isclose(sum(decomposition),total,atol=1e-5,rtol=1e-8),\
        f"Bad FP/FN/TP accounting: {sum(decomposition)} vs {total}"
    norm_per_sample=(err/SCALE).mean(1)
    macro=float(np.mean([norm_per_sample[COMP[idx]==k].mean() for k in range(8)]))
    pure=P[idx].sum(1)==1;mix=P[idx].sum(1)>=2
    mask_tp=tp
    conditional=float(np.mean(np.abs(np.log10(chat[mask_tp])-np.log10(real[mask_tp])))) if mask_tp.any() else np.nan
    out=dict(e2e_macro8_nmae=macro,pooled_mae_ppb=float(err.mean()),
             pure_nmae=float(norm_per_sample[pure].mean()),
             mixture_nmae=float(norm_per_sample[mix].mean()),
             fn_error_ppb_sum=decomposition[0],fp_error_ppb_sum=decomposition[1],
             tp_error_ppb_sum=decomposition[2],fn_fraction=decomposition[0]/total if total else 0.,
             fp_fraction=decomposition[1]/total if total else 0.,
             tp_fraction=decomposition[2]/total if total else 0.,
             tp_log_mae=conditional)
    return out

def recipe_macro_bce(idx,prob):
    y=P[idx].astype(float)
    p=np.clip(prob,1e-7,1-1e-7)
    sample=-np.mean(y*np.log(p)+(1-y)*np.log(1-p),axis=1)
    return float(np.mean([sample[COMP[idx]==k].mean() for k in range(8)]))

def select_global_threshold(prob_val):
    # Only validation, 7 predetermined candidates. Ties prefer mix EM, then 0.5.
    choices=[]
    for t in (.20,.30,.40,.50,.60,.70,.80):
        z=(prob_val>=t).astype(np.uint8)
        a=class_metrics(VA,z)
        choices.append((a["macro8"],a["exact_mix"],-abs(t-.5),-t,t))
    return float(max(choices)[-1])

## 3. Cascaded systems

Compare ExtraTrees/ROCKET detectors followed by a source concentration median or an ExtraTrees quantifier. Predicted, not true, presence is passed to the quantifier.

In [ ]:
FEATURES_WITH_ORACLE=np.concatenate([F,P.astype(np.float32)],axis=1)
MED_LOG=np.empty(3)
REGRESSORS=[]
for j in range(3):
    ids=TR[P[TR,j]==1]
    t=LOG[ids,j]
    w=W_REG[ids,j].astype(float)
    order=np.argsort(t)
    MED_LOG[j]=t[order[np.searchsorted(np.cumsum(w[order]),.5*w.sum())]]
    model=ExtraTreesRegressor(n_estimators=300,min_samples_leaf=2,
                             max_features="sqrt",random_state=42,n_jobs=-1)
    model.fit(FEATURES_WITH_ORACLE[ids],t,sample_weight=w/w.mean())
    REGRESSORS.append(model)

def gas_median_log(z):
    return np.broadcast_to(MED_LOG,(len(z),3)).copy()

def et_reg_log(z,idx):
    # Crucial: at inference, the regressor sees ONLY detected/predicted presence.
    z=z.astype(np.float32)
    fx=np.concatenate([F[idx],z],axis=1)
    out=np.zeros((len(idx),3),dtype=float)
    for j,model in enumerate(REGRESSORS):
        positive=np.flatnonzero(z[:,j]>0)
        if len(positive):out[positive,j]=model.predict(fx[positive])
    return out

detectors={}
timings=[]
t0=time.perf_counter()
forests=[]
for j in range(3):
    model=ExtraTreesClassifier(n_estimators=300,min_samples_leaf=1,
                               max_features="sqrt",random_state=42,n_jobs=-1)
    model.fit(F[TR],P[TR,j].astype(int),sample_weight=W_CLS[TR])
    forests.append(model)
detectors[("ExtraTrees3",42)]=np.column_stack([model.predict_proba(F)[:,1] for model in forests])
timings.append(dict(model="ExtraTrees3",seed=42,seconds=time.perf_counter()-t0))

if RUN_ROCKET:
    try:
        from sktime.transformations.panel.rocket import Rocket
    except ImportError as exc:
        raise ImportError("Full E7 requires sktime + numba: pip install sktime numba") from exc
    t0=time.perf_counter()
    rk=Rocket(num_kernels=3000,normalise=False,n_jobs=1,random_state=42)
    phi_tr=np.asarray(rk.fit_transform(XN[TR]),dtype=np.float32)
    sc=StandardScaler(with_mean=False).fit(phi_tr)
    phi_tr=sc.transform(phi_tr)
    rcs=[]
    for j in range(3):
        reg=RidgeClassifier(alpha=1.,solver="lsqr")
        reg.fit(phi_tr,P[TR,j],sample_weight=W_CLS[TR])
        rcs.append(reg)
    # Transform in fixed chunks; no refitting outside TR.
    vals=[]
    for ids in np.array_split(np.arange(len(X)),max(1,int(np.ceil(len(X)/256)))):
        ft=sc.transform(np.asarray(rk.transform(XN[ids]),dtype=np.float32))
        vals.append(np.column_stack([expit(reg.decision_function(ft)) for reg in rcs]))
    detectors[("ROCKET",42)]=np.vstack(vals)
    timings.append(dict(model="ROCKET",seed=42,seconds=time.perf_counter()-t0))

print("Fitted detectors:",list(detectors),"median (log ppb):",MED_LOG.round(3))

## 4. Matched multi-task comparison

Use the same CNN encoder, presence head, seeds, and selection criterion with or without auxiliary concentration loss. Compare the joint regression head with source-median concentration estimates at a fixed detector.

In [ ]:
class SharedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(
            nn.Conv1d(12,32,7,padding=3),nn.BatchNorm1d(32),nn.GELU(),nn.MaxPool1d(2),
            nn.Conv1d(32,64,5,padding=2),nn.BatchNorm1d(64),nn.GELU(),nn.MaxPool1d(2),
            nn.Conv1d(64,96,3,padding=1),nn.BatchNorm1d(96),nn.GELU())
        self.shared=nn.Sequential(nn.Linear(192,96),nn.GELU(),nn.Dropout(.20))
        self.presence=nn.Linear(96,3)
        self.regression=nn.Linear(96,3)
    def forward(self,x):
        y=self.encoder(x)
        h=self.shared(torch.cat([y.mean(-1),y.amax(-1)],dim=1))
        return self.presence(h),self.regression(h)

def fit_cnn(mode,seed):
    assert mode in ("presence_only","joint")
    seeded(seed)
    model=SharedCNN().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-4)
    targets=torch.from_numpy(LOG[TR].astype(np.float32))
    data=TensorDataset(torch.from_numpy(XN[TR]),
        torch.from_numpy(P[TR].astype(np.float32)),targets,
        torch.from_numpy(W_CLS[TR]),
        torch.from_numpy(W_REG[TR]))
    def predict(ids):
        model.eval();probs=[];logc=[]
        with torch.no_grad():
            for a in np.array_split(ids,max(1,int(np.ceil(len(ids)/256)))):
                xb=torch.as_tensor(XN[a],device=DEVICE)
                pr,rg=model(xb)
                probs.append(torch.sigmoid(pr).cpu().numpy())
                logc.append(rg.cpu().numpy())
        return np.vstack(probs),np.vstack(logc)
    best=float("inf");best_state=None;best_epoch=0;stale=0
    rows=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();batch_losses=[]
        loader=DataLoader(data,batch_size=BATCH,shuffle=True,num_workers=0,
                          generator=torch.Generator().manual_seed(seed+epoch))
        for x,p,y,wc,wr in loader:
            x,p,y,wc,wr=(v.to(DEVICE) for v in (x,p,y,wc,wr))
            opt.zero_grad()
            logits,logs=model(x)
            bce=nn.functional.binary_cross_entropy_with_logits(logits,p,reduction="none").mean(1)
            cls_loss=(bce*wc).sum()/wc.sum().clamp_min(1e-6)
            reg=nn.functional.smooth_l1_loss(logs,y,reduction="none",beta=.15)
            reg_loss=(reg*wr).sum()/wr.sum().clamp_min(1e-6)
            loss=cls_loss+(LAMBDA*reg_loss if mode=="joint" else 0.*reg_loss)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),5.)
            opt.step()
            batch_losses.append(float(loss.detach().cpu()))
        vp,_=predict(VA)
        score=recipe_macro_bce(VA,vp)
        rows.append(dict(model=mode,seed=seed,epoch=epoch,train_loss=float(np.mean(batch_losses)),
                         validation_macro_bce=score))
        if score<best-1e-5:
            best,best_epoch,best_state,stale=score,epoch,copy.deepcopy(model.state_dict()),0
        else:stale+=1
        if stale>=PATIENCE:break
    assert best_state is not None
    model.load_state_dict(best_state)
    p,r=predict(np.arange(len(X)))
    torch.save(dict(state_dict={k:v.cpu() for k,v in best_state.items()},
        train_mean=MEAN,train_std=STD,mode=mode,seed=seed,
        selected_epoch=best_epoch,lambda_reg=LAMBDA if mode=="joint" else 0.),
        OUT/f"CNN_{mode}_{seed}.pt")
    print("CNN",mode,seed,"epoch",best_epoch,"val macro BCE",round(best,4))
    del model
    if DEVICE.type=="cuda":torch.cuda.empty_cache()
    return p,r,rows,best_epoch

joint_regressions={}
history=[]
if RUN_NEURAL:
    assert SharedCNN()(torch.zeros(2,12,200))[0].shape==(2,3)
    for mode in ("presence_only","joint"):
        for seed in SEEDS:
            t0=time.perf_counter()
            proba,logs,hist,epoch=fit_cnn(mode,seed)
            detectors[(f"CNN_{mode}",seed)]=proba
            if mode=="joint":joint_regressions[seed]=logs
            history.extend(hist)
            timings.append(dict(model=f"CNN_{mode}",seed=seed,seconds=time.perf_counter()-t0,
                                best_epoch=epoch))
pd.DataFrame(history).to_csv(OUT/"training_history.csv",index=False)
pd.DataFrame(timings).to_csv(OUT/"model_costs.csv",index=False)

## 5. Future-period evaluation

Freeze the validation-selected thresholds before scoring either future test.

In [ ]:
RUNS={}
decision_rows=[]
def register(name,seed,z,logc,threshold=np.nan,source=""):
    assert z.shape==Y.shape and logc.shape==Y.shape
    z=z.astype(np.uint8)
    chat=pred_ppb(z,logc)
    assert np.all(chat[z==0]==0.)
    RUNS[(name,seed)]=dict(z=z,chat=chat,logc=logc,threshold=threshold,source=source)

mode_id=int(np.bincount(COMP[TR],minlength=8).argmax())
for name,comp_id in [("AlwaysAir",0),("TrainMode",mode_id),("AlwaysAll",7)]:
    z=np.tile(BITS[comp_id],(len(D),1))
    register(name,42,z,gas_median_log(z),source="signal_free")

for (det,seed),prob in detectors.items():
    thr=select_global_threshold(prob[VA])
    z=(prob>=thr).astype(np.uint8)
    decision_rows.append(dict(detector=det,seed=seed,global_threshold=thr,
                              val_macro8=class_metrics(VA,z[VA])["macro8"]))
    register(det+"+Median",seed,z,gas_median_log(z),thr,"source_median")
    if det in ("ExtraTrees3","ROCKET"):
        all_logs=np.zeros_like(Y,dtype=float)
        for split,idx in SPLITS.items():
            all_logs[idx]=et_reg_log(z[idx],idx)
        register(det+"+ExtraTreesReg",seed,z,all_logs,thr,"predicted_presence_to_regressor")
    if det=="CNN_joint":
        register(det+"+RegHead",seed,z,joint_regressions[seed],thr,"joint_no_oracle")

pd.DataFrame(decision_rows).to_csv(OUT/"thresholds.csv",index=False)

score_rows=[];pred_rows=[];gas_rows=[];comp_rows=[]
for (name,seed),out in RUNS.items():
    z,chat=out["z"],out["chat"]
    for split,ids in SPLITS.items():
        cm=class_metrics(ids,z[ids]);em=e2e_metrics(ids,z[ids],chat[ids])
        score_rows.append(dict(model=name,seed=seed,split=split,n=len(ids),
                               threshold=out["threshold"],**cm,**em))
        err=np.abs(chat[ids]-Y[ids])
        for j,g in enumerate(GASES):
            p=P[ids,j].astype(bool);d=z[ids,j].astype(bool)
            gas_rows.append(dict(model=name,seed=seed,split=split,gas=g,
                n=len(ids),n_true_positive=int(p.sum()),n_pred_positive=int(d.sum()),
                mae_ppb=float(err[:,j].mean()),
                fn_count=int((p&~d).sum()),fp_count=int((~p&d).sum()),
                tp_mae_ppb=float(err[p&d,j].mean()) if (p&d).any() else np.nan))
        for k in range(8):
            mask=COMP[ids]==k
            comp_rows.append(dict(model=name,seed=seed,split=split,composition=k,n=int(mask.sum()),
                exact_match=float(np.mean(np.all(z[ids][mask]==P[ids][mask],axis=1))),
                normalized_mae=float(np.mean((err[mask]/SCALE).mean(1)))))
        for jj,i in enumerate(ids):
            row=dict(model=name,seed=seed,split=split,sample_id=D.iloc[i].sample_id,
                     day=str(D.iloc[i].time.date()),recipe=D.iloc[i].canonical_recipe,
                     true_composition=int(COMP[i]),pred_composition=int((z[i]*np.array([1,2,4])).sum()),
                     abs_error_ppb_sum=float(err[jj].sum()))
            for g in range(3):
                row[f"true_{GASES[g]}_ppb"]=float(Y[i,g])
                row[f"pred_{GASES[g]}_ppb"]=float(chat[i,g])
                row[f"pred_{GASES[g]}_present"]=int(z[i,g])
            pred_rows.append(row)

SCORES=pd.DataFrame(score_rows)
PREDICTIONS=pd.DataFrame(pred_rows)
GAS_REPORT=pd.DataFrame(gas_rows)
COMPOSITION_REPORT=pd.DataFrame(comp_rows)
SCORES.to_csv(OUT/"scores.csv",index=False)
PREDICTIONS.to_csv(OUT/"predictions.csv",index=False)
GAS_REPORT.to_csv(OUT/"per_gas.csv",index=False)
COMPOSITION_REPORT.to_csv(OUT/"per_composition.csv",index=False)

print("\nBOTH FUTURE TESTS (lower E2E error better):")
print(SCORES[SCORES.split.isin(["within130","future146"])]
      [["model","seed","split","e2e_macro8_nmae","pooled_mae_ppb","macro8","exact_mix",
        "fn_fraction","fp_fraction","tp_fraction"]]
      .to_string(index=False,float_format=lambda a:f"{a:.3f}"))

## 6. Recipe-matched analysis

Compare identical nominal recipes in both tests and decompose concentration errors by detection outcome. This does not isolate physical sensor aging.

In [ ]:
# Shared recipes: compare only exact mixtures represented in BOTH future periods.
mix130=TEST130[P[TEST130].sum(1)>=2]
mix146=TEST146[P[TEST146].sum(1)>=2]
shared=sorted(set(D.iloc[mix130].canonical_recipe)&set(D.iloc[mix146].canonical_recipe))
assert len(shared)>0
MATCHED=[]
for (model,seed),r in RUNS.items():
    for split,idx in [("within130",mix130),("future146",mix146)]:
        idx=idx[D.iloc[idx].canonical_recipe.isin(shared).to_numpy()]
        for key in shared:
            group=idx[D.iloc[idx].canonical_recipe.to_numpy()==key]
            if len(group)==0:continue
            err=np.abs(r["chat"][group]-Y[group])
            MATCHED.append(dict(model=model,seed=seed,split=split,recipe=key,
                n=len(group),normalized_mae=float((err/SCALE).mean(axis=1).mean()),
                exact=float(np.mean(np.all(r["z"][group]==P[group],axis=1)))))
MATCHED=pd.DataFrame(MATCHED)
MATCHED.to_csv(OUT/"exact_recipe_matched.csv",index=False)
RECIPE_SUMMARY=MATCHED.groupby(["model","seed","split"],as_index=False).agg(
    recipes=("recipe","nunique"),mean_recipe_nmae=("normalized_mae","mean"),
    mean_recipe_exact=("exact","mean"),records=("n","sum"))
RECIPE_SUMMARY.to_csv(OUT/"recipe_matched_summary.csv",index=False)

PAIRS=[("CNN_joint+Median","CNN_presence_only+Median","Auxiliary loss: detector"),
       ("CNN_joint+RegHead","CNN_joint+Median","Joint head vs median"),
       ("ExtraTrees3+ExtraTreesReg","ExtraTrees3+Median","Cascaded regressor vs median")]
if RUN_ROCKET:PAIRS.append(("ROCKET+ExtraTreesReg","ROCKET+Median","ROCKET regressor vs median"))
PAIRED=[]
for a,b,meaning in PAIRS:
    seeds=sorted(set(s for m,s in RUNS if m==a)&set(s for m,s in RUNS if m==b))
    for seed in seeds:
        for split,idx in (("within130",TEST130),("future146",TEST146)):
            err_a=(np.abs(RUNS[a,seed]["chat"][idx]-Y[idx])/SCALE).mean(1)
            err_b=(np.abs(RUNS[b,seed]["chat"][idx]-Y[idx])/SCALE).mean(1)
            mix=P[idx].sum(1)>=2
            PAIRED.append(dict(contrast=meaning,seed=seed,split=split,n=len(idx),
                delta_nmae_sample_mean=float(np.mean(err_a-err_b)),
                delta_nmae_mixture_mean=float(np.mean((err_a-err_b)[mix])),
                # negative means first method is better
                delta_macro8_nmae=float(e2e_metrics(idx,RUNS[a,seed]["z"][idx],
                            RUNS[a,seed]["chat"][idx])["e2e_macro8_nmae"]-
                        e2e_metrics(idx,RUNS[b,seed]["z"][idx],
                            RUNS[b,seed]["chat"][idx])["e2e_macro8_nmae"])))
PAIRED=pd.DataFrame(PAIRED)
PAIRED.to_csv(OUT/"paired_ablation.csv",index=False)

print("Shared exact mixture recipes:",len(shared),"of",D.iloc[mix130].canonical_recipe.nunique(),
      "within-130 and",D.iloc[mix146].canonical_recipe.nunique(),"future-146")
print("\nPaired ablations; negative delta favors first method:")
print(PAIRED[PAIRED.split=="future146"].to_string(index=False,float_format=lambda x:f"{x:.4f}"))

## 7. Exports

Retain individual predictions, paired contrasts, per-gas and per-composition errors, run settings, and manuscript-ready tables and figures.

In [ ]:
MODEL_ORDER=["AlwaysAir","TrainMode","AlwaysAll",
    "ExtraTrees3+Median","ExtraTrees3+ExtraTreesReg",
    "ROCKET+Median","ROCKET+ExtraTreesReg",
    "CNN_presence_only+Median","CNN_joint+Median","CNN_joint+RegHead"]
FUT=SCORES[SCORES.split.isin(("within130","future146"))].copy()
def show_score(model,split,key):
    x=FUT[(FUT.model==model)&(FUT.split==split)][key].to_numpy(float)
    if not len(x):return "—"
    if len(x)==1:return f"{x[0]:.3f}"
    return f"{np.mean(x):.3f} $\\pm$ {np.std(x,ddof=1):.3f}"

def tex_escape(s):return s.replace("_",r"\_")
latex=[
r"\begin{table*}[t]",r"\centering\small",
r"\caption{E7 end-to-end nominal concentration estimation without oracle inputs. "
r"Two future tests use identical fitted models: late records within card 130 (28 September) "
r"and later card 146. Macro-8 normalized MAE (nMAE, lower is better) equally weights eight "
r"true compositions, normalizing each gas by its train-only maximum nominal ppb. "
r"Macro-8 is composition recall (higher is better); Mix EM is mixture exact match. "
r"Neural uncertainties are SD across three seeds, not independent exposure days.}",
r"\label{tab:e7}",r"\begin{tabular}{@{}lrrrrr@{}}",r"\toprule",
r"Model & 130 nMAE $\downarrow$ & 146 nMAE $\downarrow$ & 130 Macro-8 $\uparrow$ "
r"& 146 Macro-8 $\uparrow$ & 146 Mix EM $\uparrow$ \\",
r"\midrule"]
for model in MODEL_ORDER:
    if model not in FUT.model.unique():continue
    latex.append(tex_escape(model)+" & "+" & ".join([
        show_score(model,"within130","e2e_macro8_nmae"),
        show_score(model,"future146","e2e_macro8_nmae"),
        show_score(model,"within130","macro8"),
        show_score(model,"future146","macro8"),
        show_score(model,"future146","exact_mix")])+r" \\")
latex.extend([r"\bottomrule",r"\end{tabular}",r"\end{table*}"])
(OUT/"E7_table.tex").write_text("\n".join(latex)+"\n",encoding="utf-8")

# Single-axes plot 1: paired end-to-end scores on each held-out cohort.
fig,ax=plt.subplots(figsize=(10,4.8))
names=[x for x in MODEL_ORDER if x in FUT.model.unique()]
ypos=np.arange(len(names))
score130=[FUT[(FUT.model==x)&(FUT.split=="within130")].e2e_macro8_nmae.mean() for x in names]
score146=[FUT[(FUT.model==x)&(FUT.split=="future146")].e2e_macro8_nmae.mean() for x in names]
ax.barh(ypos-.18,score130,height=.35,label="130 / Sep 28")
ax.barh(ypos+.18,score146,height=.35,label="146 / Oct 5–8")
ax.set_yticks(ypos,names);ax.invert_yaxis()
ax.set_xlabel("Macro-8 normalized end-to-end MAE (lower is better)")
ax.legend(frameon=False);fig.tight_layout()
fig.savefig(OUT/"E7_cohort_comparison.pdf");plt.close(fig)

# Single-axes plot 2: concentration-error contribution by mechanism.
selection=FUT[FUT.split=="future146"].groupby("model",sort=False)[
    ["fn_fraction","fp_fraction","tp_fraction"]].mean().reindex(names)
fig,ax=plt.subplots(figsize=(10,4.8))
left=np.zeros(len(names))
for col,label in [("fn_fraction","False negatives"),
                  ("fp_fraction","False positives"),("tp_fraction","True positives")]:
    values=selection[col].to_numpy(float)
    ax.barh(ypos,values,left=left,label=label)
    left+=values
ax.set_yticks(ypos,names);ax.invert_yaxis()
ax.set_xlabel("Fraction of total absolute ppb error, campaign 146")
ax.set_xlim(0,1);ax.legend(frameon=False)
fig.tight_layout();fig.savefig(OUT/"E7_error_decomposition.pdf");plt.close(fig)

complete=bool(RUN_ROCKET and RUN_NEURAL)
config=dict(task="E7 non-oracle joint quantification",complete=complete,
    train_records=len(TR),validation_records=len(VA),
    within130_records=len(TEST130),future146_records=len(TEST146),
    validation="Six frozen Sep27 hourly blocks from StratifiedGroupKFold(4) fold 0 (sklearn 1.8)",
    validation_hours=sorted(VAL_HOURS),
    model_order=MODEL_ORDER,gases=GASES,train_only_ppb_scale=SCALE.tolist(),
    neural_seeds=SEEDS,lambda_reg=LAMBDA,
    selection="grouped Sep27 val macro-composition BCE; global threshold val macro8",
    primary="macro8 normalized ppb MAE",cache_signature=SIGNATURE,
    no_oracle_at_inference=True,test_previously_inspected=True)
(OUT/"run_config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")

# Markdown report without pandas.to_markdown/tabulate dependency.
report=[
"# E7 — End-to-end composition and quantification","",
"## Scientific question",
"Can sensor-only inference recover nominal presence and concentration on future readings?",
"",
f"- Train {len(TR)}; grouped validation {len(VA)}; independent within130 test {len(TEST130)};",
f"  future146 test {len(TEST146)}. Common mixture recipes: {len(shared)}.",
f"- Complete benchmark: {complete}. Auxiliary CNN loss multiplier: {LAMBDA}.",
"- This is RETROSPECTIVE: 146 has already been examined in E5/E6.",
"- Small air/pure support on 28 September; only one mixture day in 146.",
"- No independently verified delivered concentrations. No causal identification of aging.",
"",
"## Both future tests (no test-based model selection)",
"```",
FUT[["model","seed","split","e2e_macro8_nmae","macro8","exact_mix",
     "pooled_mae_ppb","fn_fraction","fp_fraction","tp_fraction"]].to_string(index=False),
"```","",
"## Matched ablations (negative nMAE difference favors first method)",
"```",PAIRED.to_string(index=False),"```","",
"## Exact-recipe-controlled evaluation",
"```",RECIPE_SUMMARY.to_string(index=False),"```","",
"## Interpretation protocol",
"1. Compare CNN_joint+Median vs CNN_presence_only+Median for auxiliary training effect on detection.",
"2. Compare CNN_joint+RegHead vs CNN_joint+Median for numeric benefit at fixed detector.",
"3. Compare ExtraTrees3+ExtraTreesReg vs ExtraTrees3+Median for cascaded regression benefit.",
"4. Compare within130 vs future146 descriptively, not as isolated physical drift.",
"5. Inspect per_gas.csv, per_composition.csv, errors FP/FN/TP, exact_recipe_matched.csv.",
"6. Never interpret three CNN seeds as independent sensor/session replications."]
(OUT/"E7_report.md").write_text("\n".join(report)+"\n",encoding="utf-8")
print("Wrote E7 outputs to:",OUT,"| complete:",complete)